# Predicting asset failure within 90 days

**Question:** Can information available at inspection time predict `failed_90d`?

This notebook uses a chronological train/validation/test design because the intended use is prediction on future inspections. It reports ranking, calibration, and operating-point metrics; compares against prevalence baselines; and separately demonstrates the effect of the potentially post-inspection field `parts_replaced`.

### Decision rules

- Primary metric: **average precision (PR AUC)**, appropriate for an imbalanced target.
- Secondary metrics: ROC AUC and Brier score.
- Model and classification threshold are chosen using only training/validation data. The newest 20% is a locked final test.
- `asset_tag` is excluded (it is unique in this file and cannot generalize).
- `parts_replaced` is excluded from the deployable model because the dictionary says it is recorded on the work order that *closed* the inspection record. It is evaluated only as a leakage/availability sensitivity analysis.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, roc_auc_score, brier_score_loss,
    precision_recall_curve, PrecisionRecallDisplay, RocCurveDisplay,
    confusion_matrix, classification_report
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
pd.set_option("display.max_colwidth", 100)
sns.set_theme(style="whitegrid", context="notebook")

## 1. Load and validate the data

The checks below make assumptions explicit and fail early if the schema or target changes.

In [ ]:
DATA_PATH = Path("data.csv")
DICT_PATH = Path("columns.csv")

data = pd.read_csv(DATA_PATH, parse_dates=["inspected_at"])
dictionary = pd.read_csv(DICT_PATH)

display(dictionary)
print(f"Rows: {len(data):,}; columns: {data.shape[1]}")
display(data.head())

In [ ]:
expected = {
    "asset_tag", "inspected_at", "site", "vibration_rms", "bearing_temp_c",
    "ambient_temp_c", "meter_hours", "load_pct", "parts_replaced", "failed_90d"
}
assert expected == set(data.columns), f"Unexpected schema: {set(data.columns) ^ expected}"
assert data["failed_90d"].isin([0, 1]).all(), "Target must be binary"
assert data["inspected_at"].notna().all(), "Inspection times must parse"

quality = pd.DataFrame({
    "dtype": data.dtypes.astype(str),
    "missing_n": data.isna().sum(),
    "missing_pct": data.isna().mean(),
    "unique_n": data.nunique(dropna=False),
})
display(quality)
print("Duplicate full rows:", data.duplicated().sum())
print("Duplicate asset tags:", data["asset_tag"].duplicated().sum())
print("Date range:", data["inspected_at"].min(), "to", data["inspected_at"].max())
print("Failure prevalence:", f'{data["failed_90d"].mean():.2%}',
      f'({data["failed_90d"].sum():,}/{len(data):,})')

## 2. Leakage and label-maturity audit

`failed_90d` requires a complete 90-day observation window after each inspection. The dataset does not provide an extraction date or an explicit follow-up-complete flag. Therefore the newest labels may be right-censored. Before deployment, confirm that **every negative example had at least 90 days of follow-up**.

`parts_replaced` is also suspicious: it belongs to the work order that closed the inspection record, so it may not exist at the instant a prediction is required. A model that uses it answers a different, later-time question.

In [ ]:
quarterly = (data.assign(quarter=data["inspected_at"].dt.to_period("Q").astype(str))
             .groupby("quarter")["failed_90d"]
             .agg(n="size", failures="sum", failure_rate="mean"))
parts_audit = data.groupby("parts_replaced")["failed_90d"].agg(n="size", failures="sum", failure_rate="mean")
display(quarterly.style.format({"failure_rate": "{:.1%}"}))
display(parts_audit.style.format({"failure_rate": "{:.1%}"}))

fig, ax = plt.subplots(figsize=(10, 4))
quarterly["failure_rate"].plot(kind="bar", ax=ax, color="#3b82f6")
ax.axhline(data["failed_90d"].mean(), color="black", ls="--", lw=1, label="overall")
ax.set(title="Failure rate changes sharply over calendar time", ylabel="Failure rate", xlabel="Inspection quarter")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Exploratory signal

These are descriptive comparisons, not causal effects. The test set remains untouched by model fitting and threshold choice.

In [ ]:
numeric = ["vibration_rms", "bearing_temp_c", "ambient_temp_c", "meter_hours", "load_pct"]
summary = data.groupby("failed_90d")[numeric].agg(["mean", "median"])
display(summary.T)

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for col, ax in zip(numeric, axes.flat):
    sns.boxplot(data=data, x="failed_90d", y=col, ax=ax, showfliers=False, color="#93c5fd")
axes.flat[-1].axis("off")
fig.suptitle("Inspection-time features by outcome", y=1.02)
plt.tight_layout()
plt.show()

## 4. Feature engineering and chronological split

The cyclical month/hour features are known at inspection time. The raw date is not used, avoiding a direct calendar trend that cannot extrapolate. The split is 60% / 20% / 20% after sorting by time. This is stricter than a random split and better represents forward deployment.

In [ ]:
data = data.sort_values("inspected_at").reset_index(drop=True)

def make_features(frame, include_parts=False):
    x = frame[["site", "vibration_rms", "bearing_temp_c", "ambient_temp_c",
               "meter_hours", "load_pct"]].copy()
    dt = frame["inspected_at"]
    x["month_sin"] = np.sin(2 * np.pi * dt.dt.month / 12)
    x["month_cos"] = np.cos(2 * np.pi * dt.dt.month / 12)
    x["hour_sin"] = np.sin(2 * np.pi * dt.dt.hour / 24)
    x["hour_cos"] = np.cos(2 * np.pi * dt.dt.hour / 24)
    if include_parts:
        x["parts_replaced"] = frame["parts_replaced"]
    return x

y = data["failed_90d"].astype(int)
X = make_features(data)
n = len(data)
train_end, valid_end = int(0.60 * n), int(0.80 * n)
idx_train = np.arange(0, train_end)
idx_valid = np.arange(train_end, valid_end)
idx_test = np.arange(valid_end, n)

split_rows = []
for name, idx in [("train", idx_train), ("validation", idx_valid), ("test", idx_test)]:
    split_rows.append({
        "split": name, "n": len(idx), "start": data.loc[idx, "inspected_at"].min(),
        "end": data.loc[idx, "inspected_at"].max(), "failures": int(y.iloc[idx].sum()),
        "prevalence": y.iloc[idx].mean()
    })
splits = pd.DataFrame(split_rows).set_index("split")
display(splits.style.format({"prevalence": "{:.2%}"}))

## 5. Candidate models and validation-only selection

The dummy model supplies a no-signal reference. The logistic model tests a mostly linear relationship; tree ensembles allow nonlinearities and interactions. Hyperparameters are deliberately modest to limit overfitting. No row from the final test is used for selection.

In [ ]:
categorical = ["site"]
numerical = [c for c in X.columns if c not in categorical]

preprocess = ColumnTransformer([
    ("site", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
    ("numeric", StandardScaler(), numerical),
])

models = {
    "dummy": DummyClassifier(strategy="prior"),
    "logistic": LogisticRegression(max_iter=2_000, random_state=RANDOM_STATE),
    "random_forest": RandomForestClassifier(
        n_estimators=400, min_samples_leaf=15, max_features=0.7,
        n_jobs=-1, random_state=RANDOM_STATE
    ),
    "gradient_boosting": GradientBoostingClassifier(
        n_estimators=100, max_depth=2, min_samples_leaf=15, random_state=RANDOM_STATE
    ),
}

fitted = {}
validation_rows = []
for name, estimator in models.items():
    pipe = Pipeline([("preprocess", clone(preprocess)), ("model", estimator)])
    pipe.fit(X.iloc[idx_train], y.iloc[idx_train])
    p = pipe.predict_proba(X.iloc[idx_valid])[:, 1]
    fitted[name] = pipe
    validation_rows.append({
        "model": name,
        "average_precision": average_precision_score(y.iloc[idx_valid], p),
        "roc_auc": roc_auc_score(y.iloc[idx_valid], p),
        "brier": brier_score_loss(y.iloc[idx_valid], p),
        "mean_prediction": p.mean(),
    })

validation_results = pd.DataFrame(validation_rows).set_index("model").sort_values("average_precision", ascending=False)
display(validation_results.style.format("{:.3f}"))

eligible = validation_results.drop(index="dummy")
selected_name = eligible["average_precision"].idxmax()
selected = fitted[selected_name]
print("Selected by validation average precision:", selected_name)

## 6. Locked future-test evaluation

Average precision should be compared with the test prevalence (the expected AP of an uninformative ranking). Brier score evaluates probability error; a constant prediction equal to the training prevalence is included because temporal prevalence drift can make otherwise reasonable probabilities poorly calibrated.

In [ ]:
test_rows = []
test_probabilities = {}
for name, pipe in fitted.items():
    p = pipe.predict_proba(X.iloc[idx_test])[:, 1]
    test_probabilities[name] = p
    test_rows.append({
        "model": name,
        "average_precision": average_precision_score(y.iloc[idx_test], p),
        "roc_auc": roc_auc_score(y.iloc[idx_test], p) if p.std() > 0 else 0.5,
        "brier": brier_score_loss(y.iloc[idx_test], p),
        "mean_prediction": p.mean(),
    })
test_results = pd.DataFrame(test_rows).set_index("model").sort_values("average_precision", ascending=False)
display(test_results.style.format("{:.3f}"))
print(f"Test prevalence / no-skill AP: {y.iloc[idx_test].mean():.3f}")
print(f"Preselected model: {selected_name}")

In [ ]:
# Bootstrap uncertainty for the preselected model (resampling test rows).
def bootstrap_metrics(y_true, prob, n_boot=1000, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    y_arr, p_arr = np.asarray(y_true), np.asarray(prob)
    values = []
    for _ in range(n_boot):
        take = rng.integers(0, len(y_arr), len(y_arr))
        yy, pp = y_arr[take], p_arr[take]
        if yy.min() == yy.max():
            continue
        values.append((average_precision_score(yy, pp), roc_auc_score(yy, pp)))
    values = np.asarray(values)
    return pd.DataFrame({
        "estimate": [average_precision_score(y_arr, p_arr), roc_auc_score(y_arr, p_arr)],
        "ci_2.5%": np.quantile(values, .025, axis=0),
        "ci_97.5%": np.quantile(values, .975, axis=0),
    }, index=["average_precision", "roc_auc"])

p_test = test_probabilities[selected_name]
display(bootstrap_metrics(y.iloc[idx_test], p_test).style.format("{:.3f}"))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
PrecisionRecallDisplay.from_predictions(y.iloc[idx_test], p_test, name=selected_name, ax=axes[0])
axes[0].axhline(y.iloc[idx_test].mean(), color="black", ls="--", label="no-skill")
axes[0].legend()
RocCurveDisplay.from_predictions(y.iloc[idx_test], p_test, name=selected_name, ax=axes[1])
plt.tight_layout()
plt.show()

### One possible operating threshold

Thresholds depend on the cost of a missed failure versus an unnecessary intervention. As an illustration, the code chooses the validation-set threshold maximizing F2 (recall weighted twice as heavily as precision), then applies that fixed threshold once to the test set.

In [ ]:
p_valid = fitted[selected_name].predict_proba(X.iloc[idx_valid])[:, 1]
precision, recall, thresholds = precision_recall_curve(y.iloc[idx_valid], p_valid)
f2 = 5 * precision[:-1] * recall[:-1] / np.maximum(4 * precision[:-1] + recall[:-1], 1e-12)
threshold = thresholds[np.nanargmax(f2)]
pred_test = (p_test >= threshold).astype(int)

print(f"Validation-selected F2 threshold: {threshold:.3f}")
print("Test confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y.iloc[idx_test], pred_test))
print(classification_report(y.iloc[idx_test], pred_test, digits=3, zero_division=0))

## 7. What drives the selected prediction?

Permutation importance measures the loss of test average precision after shuffling a feature. It is model-specific and can split importance among correlated variables; it is not a causal interpretation.

In [ ]:
importance = permutation_importance(
    selected, X.iloc[idx_test], y.iloc[idx_test], scoring="average_precision",
    n_repeats=20, random_state=RANDOM_STATE, n_jobs=-1
)
importance_df = (pd.DataFrame({
    "feature": X.columns,
    "mean_ap_decrease": importance.importances_mean,
    "std": importance.importances_std,
}).sort_values("mean_ap_decrease", ascending=False))
display(importance_df)

fig, ax = plt.subplots(figsize=(8, 4.5))
plot_df = importance_df.sort_values("mean_ap_decrease")
ax.barh(plot_df["feature"], plot_df["mean_ap_decrease"], xerr=plot_df["std"], color="#60a5fa")
ax.axvline(0, color="black", lw=1)
ax.set(xlabel="Decrease in test average precision when shuffled", title="Permutation importance")
plt.tight_layout()
plt.show()

## 8. Sensitivity analyses

### A. Random split (optimism diagnostic)

This mixes inspection dates and is **not** the deployment estimate. It shows how much more optimistic an ordinary random holdout can look when calendar regimes overlap.

In [ ]:
train_idx_r, test_idx_r = train_test_split(
    np.arange(len(data)), test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
random_pipe = clone(selected)
random_pipe.fit(X.iloc[train_idx_r], y.iloc[train_idx_r])
p_random = random_pipe.predict_proba(X.iloc[test_idx_r])[:, 1]
random_result = pd.Series({
    "average_precision": average_precision_score(y.iloc[test_idx_r], p_random),
    "roc_auc": roc_auc_score(y.iloc[test_idx_r], p_random),
    "brier": brier_score_loss(y.iloc[test_idx_r], p_random),
    "prevalence": y.iloc[test_idx_r].mean(),
})
display(random_result.to_frame("random_holdout").style.format("{:.3f}"))

### B. Including `parts_replaced` (availability/leakage diagnostic)

This result must **not** be interpreted as an inspection-time model unless the business process confirms that the value is finalized before scoring. The cross-tab above already suggests near-deterministic target encoding.

In [ ]:
X_parts = make_features(data, include_parts=True)
num_parts = [c for c in X_parts.columns if c != "site"]
preprocess_parts = ColumnTransformer([
    ("site", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["site"]),
    ("numeric", StandardScaler(), num_parts),
])
parts_pipe = Pipeline([
    ("preprocess", preprocess_parts),
    ("model", GradientBoostingClassifier(
        n_estimators=100, max_depth=2, min_samples_leaf=15, random_state=RANDOM_STATE
    )),
])
parts_pipe.fit(X_parts.iloc[idx_train], y.iloc[idx_train])
p_parts = parts_pipe.predict_proba(X_parts.iloc[idx_test])[:, 1]
parts_result = pd.Series({
    "average_precision": average_precision_score(y.iloc[idx_test], p_parts),
    "roc_auc": roc_auc_score(y.iloc[idx_test], p_parts),
    "brier": brier_score_loss(y.iloc[idx_test], p_parts),
    "mean_prediction": p_parts.mean(),
})
display(parts_result.to_frame("with_parts_replaced").style.format("{:.3f}"))

## 9. Conclusion and recommendation

The notebook intentionally distinguishes **signal** from **deployment readiness**:

1. Compare the preselected model's test average precision with test prevalence and inspect its confidence interval. Material lift indicates some inspection-time ranking signal; a wide interval or near-baseline result means the evidence is weak.
2. Compare validation and test results. A large drop is evidence that the relationship is not stable across time, even if a random split looks good.
3. Do not use `parts_replaced` unless it is provably known at the scoring instant. Its near-deterministic association with the target is a classic leakage warning.
4. Do not operationalize probabilities until label maturity is verified and probabilities are recalibrated on a recent, representative cohort. The sharp fall in quarterly prevalence is consistent with either real drift or incomplete 90-day follow-up.

**Recommended next gate:** obtain an extraction date or `followup_complete_90d` flag, remove immature inspections, define the scoring timestamp and feature-availability contract, then repeat rolling-origin validation over several mature quarters. Choose an alert threshold from explicit missed-failure and inspection/intervention costs. Only then run a prospective shadow deployment.

In [ ]:
# Compact, data-derived verdict for this run.
selected_test = test_results.loc[selected_name]
baseline_ap = y.iloc[idx_test].mean()
lift = selected_test["average_precision"] / baseline_ap
val_ap = validation_results.loc[selected_name, "average_precision"]

print(f"Selected model: {selected_name}")
print(f"Validation AP: {val_ap:.3f}")
print(f"Future-test AP: {selected_test['average_precision']:.3f} vs baseline {baseline_ap:.3f} ({lift:.1f}x)")
print(f"Future-test ROC AUC: {selected_test['roc_auc']:.3f}")
print(f"Future-test mean predicted risk: {selected_test['mean_prediction']:.3f}; observed: {baseline_ap:.3f}")
if selected_test["roc_auc"] < 0.65 or selected_test["average_precision"] < 2 * baseline_ap:
    print("VERDICT: Not enough stable out-of-time evidence for deployment as currently defined.")
else:
    print("VERDICT: Useful ranking signal exists, but deployment still requires mature labels and recent calibration.")